# CUDA Sparse Matrix-Vector Multiplication (SpMV) — Team 7
## Academic HPC Benchmarking, GPU Profiling, and Architectural Evaluation

**Team 7** | High Performance Computing

This Google Colab notebook implements and benchmarks 5 SpMV approaches on NVIDIA GPUs:
1. **Scalar CSR CUDA Kernel** (1 thread per row)
2. **Vector CSR CUDA Kernel** (1 warp / 32 threads per row with intra-warp shuffle reduction)
3. **ELLPACK CUDA Kernel** (Column-major layout with zero branch divergence & 100% coalescing)
4. **HYB CUDA Kernel** (Hybrid ELL + COO with atomicAdd accumulation)
5. **NVIDIA cuSPARSE SpMV Reference** (Vendor-tuned production baseline)
6. **Sequential CPU Baseline** (C++ -O3 baseline for speedup comparison)

Evaluated across 13 SuiteSparse matrices spanning regular banded, structural, CFD, and scale-free power-law graphs.

### Step 1: GPU and Environment Verification

In [ ]:
# 1. Verify NVIDIA GPU Availability
!nvidia-smi

# 2. Check CUDA Compiler
!nvcc --version

# 3. Verify cuSPARSE Header and Library
!find /usr/local/cuda* -name cusparse.h 2>/dev/null

### Step 2: Repository Setup and Session Persistence
Handles Colab session resets gracefully and supports Google Drive backup.

In [ ]:
import os
from pathlib import Path

# Mount Google Drive if desired to persist raw benchmarks across Colab resets
PERSIST_TO_DRIVE = False  # Set to True if mounting Google Drive
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# Ensure workspace directory
WORKSPACE = Path.cwd()
print(f"Current working directory: {WORKSPACE}")

# Ensure output directories exist
os.makedirs("bin", exist_ok=True)
os.makedirs("results/raw", exist_ok=True)
os.makedirs("results/summary", exist_ok=True)
os.makedirs("results/plots", exist_ok=True)
os.makedirs("report", exist_ok=True)

### Step 3: Hardware Specification Capture

In [ ]:
!chmod +x scripts/collect_hardware.sh
!./scripts/collect_hardware.sh
!cat results/hardware_info.txt

### Step 4: Build CPU and CUDA Implementations
Compiles the CPU baseline (-O3) and all 5 CUDA SpMV implementations using nvcc.

In [ ]:
!chmod +x scripts/build.sh
!./scripts/build.sh

### Step 5: Strict Correctness Validation
Validates all implementations against reference outputs in `reference_outputs/`.

In [ ]:
# Run deterministic test matrix
!./bin/cpu_spmv --matrix datasets/original/test_small.mtx --vector datasets/vectors/test_small_x.txt --reference reference_outputs/test_small_y_reference.txt --dataset test_small
!./bin/scalar_csr --matrix datasets/original/test_small.mtx --vector datasets/vectors/test_small_x.txt --reference reference_outputs/test_small_y_reference.txt --dataset test_small
!./bin/vector_csr --matrix datasets/original/test_small.mtx --vector datasets/vectors/test_small_x.txt --reference reference_outputs/test_small_y_reference.txt --dataset test_small
!./bin/ell --matrix datasets/original/test_small.mtx --vector datasets/vectors/test_small_x.txt --reference reference_outputs/test_small_y_reference.txt --dataset test_small
!./bin/hyb --matrix datasets/original/test_small.mtx --vector datasets/vectors/test_small_x.txt --reference reference_outputs/test_small_y_reference.txt --dataset test_small
!./bin/cusparse_spmv --matrix datasets/original/test_small.mtx --vector datasets/vectors/test_small_x.txt --reference reference_outputs/test_small_y_reference.txt --dataset test_small

### Step 6: Full Benchmark Execution and Parameter Tuning
Runs 10 measured iterations (after warm-up) across all 13 matrices, testing thread configurations and ELL widths.

In [ ]:
!chmod +x scripts/run_benchmarks.sh
!./scripts/run_benchmarks.sh

### Step 7: GPU Profiling with NVIDIA Nsight Compute (ncu)
Checks availability of hardware performance counters and profiles memory throughput, warp efficiency, and divergence.

In [ ]:
!chmod +x scripts/profile.sh
!./scripts/profile.sh
!cat results/profiling_report.txt

### Step 8: Generate Publication-Quality Graphs
Generates all 12 required benchmark plots (Execution time, GFLOP/s, Speedup, Roofline, Memory Bandwidth, etc.).

In [ ]:
!python3 scripts/generate_plots.py

### Step 9: Inline Visualization and Performance Tables

In [ ]:
import pandas as pd
from IPython.display import Image, display

# Display Aggregated Summary Table
summary_csv = "results/summary/team7_summary.csv"
if os.path.exists(summary_csv):
    df = pd.read_csv(summary_csv)
    display(df.head(25))

# Display Key Graphs Inline
plot_files = [
    "results/plots/2_gflops_comparison.png",
    "results/plots/3_speedup_vs_cpu.png",
    "results/plots/6_roofline_plot.png",
    "results/plots/10_achieved_bandwidth.png",
    "results/plots/11_cusparse_percentage.png",
    "results/plots/12_ell_padding_overhead.png"
]

for p in plot_files:
    if os.path.exists(p):
        print(f"=== Plot: {p} ===")
        display(Image(filename=p))


### Step 10: Best Format Per Matrix Comparison Summary

| Matrix Name | Category | Rows | Nonzeros | Average Row Length | Irregularity | Recommended Format | Architectural Reason |
|:---|:---|---:|---:|---:|---:|:---|:---|
| `qpband` | Banded | 20,000 | 45,000 | 2.25 | 1.33 | **ELLPACK** | Low row-length variance, 100% coalesced access, 0% warp divergence |
| `sparsine` | Structural | 50,000 | 1,548,988 | 30.98 | 1.81 | **ELLPACK / cuSPARSE** | Minimal padding overhead (1.81x), regular mesh sparsity |
| `Goodwin_071` | CFD | 56,021 | 1,797,934 | 32.09 | 1.93 | **Vector CSR / ELL** | Warp stride matches average row length (32.09), coalesced loads |
| `appu` | Graph | 14,000 | 1,853,104 | 132.36 | 2.22 | **Vector CSR** | High average row length (132), full warp utilization without ELL overhead |
| `mixtank_new` | CFD | 29,957 | 1,995,041 | 66.60 | 2.31 | **Vector CSR** | Multiple 32-thread iterations per row, intra-warp reduction |
| `bbmat` | CFD | 38,744 | 1,771,722 | 45.73 | 2.76 | **Vector CSR** | Divergence-free warp execution across irregular blocks |
| `ns3Da` | CFD | 20,414 | 1,679,599 | 82.28 | 3.72 | **Vector CSR** | Dense regular rows, optimal memory bandwidth utilization |
| `bcsstk35` | Structural | 30,237 | 1,450,163 | 47.96 | 4.63 | **Vector CSR / cuSPARSE** | Good cache locality, warp reduction reduces register pressure |
| `wy2010` | Graph | 86,204 | 427,586 | 4.96 | 27.42 | **HYB (ELL+COO)** | Short average row length with sparse long-row outliers |
| `av41092` | 2D/3D FEM | 41,092 | 1,683,902 | 40.98 | 52.10 | **HYB (ELL+COO)** | Avoids 52x ELL padding penalty; atomicAdd handles overflow rows |
| `soc_sign_slashdot` | Power-Law | 82,144 | 549,202 | 6.69 | 64.02 | **HYB (ELL+COO)** | Power-law distribution: ELL for regular nodes, COO for hubs |
| `TSOPF_FS_b39_c19` | Power Grid | 76,216 | 1,977,600 | 25.95 | 1465.43 | **HYB / Vector CSR** | Pure ELL requires 34GB RAM (1465x padding). HYB isolates 38k hub row |